## This note shows how to use sklearn to set up a regression for p(z)

We are going to extract "features" and "targets" from both the test and train data sets, and use those
to: 

1. Train an Estimator or "Regression model" using the training data
2. Look at how well it performs on the test data

#### Do the usual imports

In [ ]:
import os
import tables_io
import numpy as np
import matplotlib.pyplot as plt
import plotting_functions
import utility_functions

#### Download and uncompress the data

In [ ]:
_ = utility_functions.setup_data()

#### Change this to match the correct location

In [ ]:
pz_dir = '.'

#### Load the test/ train data

In [ ]:
train = tables_io.read(f"{pz_dir}/data/dp1_matched_v4_train.hdf5")
test = tables_io.read(f"{pz_dir}/data/dp1_matched_v4_test.hdf5")
#d.keys()
#train = tables_io.sliceObj(d, slice(0, -1, 10))
#test = tables_io.sliceObj(d, slice(1, -1, 10))

In [ ]:
# Build the feature matrix: total magnitude + 5 adjacent-band colors (u-g, g-r, r-i, i-z, z-y).
# Colors carry the photo-z signal; total magnitude disambiguates degenerate color combinations.
# gaap1p0Mag: Gaussian Aperture and PSF magnitude in a 1.0 arcsec aperture —
# more uniform across objects than PSF mags, good for color comparisons.
train_targets, train_features = utility_functions.prepare_data_total_mag_and_colors(train, '{band}_gaap1p0Mag', 'ugrizy')
test_targets, test_features = utility_functions.prepare_data_total_mag_and_colors(test, '{band}_gaap1p0Mag', 'ugrizy')

Do PCA for in case we want to use it

In [ ]:
from sklearn import preprocessing
from sklearn.decomposition import PCA

In [ ]:
# StandardScaler: subtract the mean and divide by std so all features have unit variance.
# This prevents features with larger numerical ranges from dominating the regressor.
# PCA: rotates into the eigenbasis of the covariance matrix.
# whiten=False keeps the PCA components at their natural variances (not normalized to 1).
scaler = preprocessing.StandardScaler().fit(train_features)
pca = PCA(n_components=train_features.shape[-1], whiten=False)

In [ ]:
# Clip at ±5σ before PCA to avoid extreme outliers (failed photometry) rotating the eigenvectors.
scaled_train = scaler.transform(train_features).clip(-5, 5)
pca_train = pca.fit(scaled_train)

In [ ]:
scaled_test = scaler.transform(test_features).clip(-5, 5)
pca_test = pca.fit(scaled_test)

Do all the regression

In [ ]:
from sklearn.ensemble import (HistGradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import HuberRegressor, LinearRegression, QuantileRegressor
from sklearn.svm import NuSVR
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor

In [ ]:
# A variety of sklearn regressors to compare.
# hbr: Histogram Gradient Boosting — very fast, handles NaNs, often best accuracy.
# etr: Extra Trees — random feature splits, lower variance than random forests.
# abr: AdaBoost — sequential boosting; can overfit noisy data.
# gpr: Gaussian Process — slow on large datasets, but returns uncertainty estimates.
# knr: K-Nearest Neighbors — analogous to template matching; interpretable.
# rnr: Radius Neighbors — like KNN but with a fixed search radius instead of k.
hbr = HistGradientBoostingRegressor()
etr = ExtraTreesRegressor()
abr = AdaBoostRegressor()
gpr = GaussianProcessRegressor()
isr = IsotonicRegression()
hur = HuberRegressor()
lir = LinearRegression()
qur = QuantileRegressor()
nsr = NuSVR()
knr = KNeighborsRegressor()
rnr = RadiusNeighborsRegressor()

In [ ]:
def run_it(reg):
    # Train on PCA-transformed features; pca_train was fit on the training set.
    # plot_biweight_stats_v_redshift shows bias and scatter as a function of true z,
    # revealing where the estimator struggles (typically at z > 1 with sparse training data).
    preds = utility_functions.run_regression(reg, scaled_train, train_targets, scaled_test)
    _ = plotting_functions.plot_true_predict_fancy(test_targets, np.nan_to_num(preds, -0.4))
    _ = plotting_functions.plot_biweight_stats_v_redshift(test_targets, preds)

In [ ]:
run_it(knr)